# Lab 01: Capstone Architecture Design

Design the architecture for a production AI agent system. Identify the
components (FastAPI, LangGraph, ChromaDB, LangFuse + PostgreSQL),
map data flows, and determine monitoring points.

No external packages required — standard library only.

In [ ]:
import os
import json
import shutil
from typing import Dict, List

WORKDIR = "/tmp/capstone-lab-15-01"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

score = 0
total = 0

## Step 1: Production AI Agent System -- Components

A production AI agent system has four application components:

| Component | Role |
|-----------|------|
| FastAPI | HTTP API gateway -- receives requests, returns responses, exposes health endpoints |
| LangGraph Agent | Stateful workflow engine -- orchestrates reasoning, tool calls, and decision logic |
| ChromaDB | Vector store -- stores and retrieves document embeddings for RAG |
| LangFuse (+ PostgreSQL) | Observability platform -- traces agent runs, tracks token usage, cost, quality, latency (backed by PostgreSQL for persistence) |

## Step 2: Data Flow Architecture

Request flow through the system:

```
User Request
     |
     v
[FastAPI]  --traces-->  [LangFuse]  (via CallbackHandler)
     |
     v
[LangGraph Agent]  --traces-->  [LangFuse]
     |        |
     |        +--retrieve-->  [ChromaDB]
     |
     v
[LLM API (Groq)]
     |
     v
Response to User
```

Key data flows:
1. User -> FastAPI -> LangGraph -> LLM -> FastAPI -> User
2. LangGraph -> ChromaDB (RAG retrieval)
3. FastAPI -> LangFuse (request traces via callback)
4. LangGraph -> LangFuse (agent traces/spans/generations)

## Step 3: Component Startup Order (Dependency Graph)

Components must start in dependency order:

- **Level 0 (no deps):** ChromaDB, LangFuse (+ PostgreSQL)
- **Level 1 (needs L0):** LangGraph Agent (needs ChromaDB, LangFuse)
- **Level 2 (needs L1):** FastAPI (needs LangGraph Agent)

Python startup sequencing controls this ordering:
```
langfuse-db -> langfuse -> support-agent
chromadb -> support-agent
```

## TODO 1: Map each component to its primary role

In [ ]:
component_roles = {
    "fastapi":    "api_gateway",
    "langgraph":  "workflow_engine",
    "chromadb":   "vector_store",
    "langfuse":   "observability_platform",
}

In [ ]:
total += 1
expected_roles = {
    "fastapi":    "api_gateway",
    "langgraph":  "workflow_engine",
    "chromadb":   "vector_store",
    "langfuse":   "observability_platform",
}
if component_roles == expected_roles:
    score += 1
    print("[PASS] Component-to-role mapping is correct")
else:
    print("[FAIL] Expected:", expected_roles)
    print("       Got:     ", component_roles)

## TODO 2: Define the correct startup order levels

In [ ]:
startup_order = {
    "level_0": ["chromadb", "langfuse"],
    "level_1": ["langgraph"],
    "level_2": ["fastapi"],
}

In [ ]:
total += 1
expected_order = {
    "level_0": ["chromadb", "langfuse"],
    "level_1": ["langgraph"],
    "level_2": ["fastapi"],
}
order_ok = True
for level in expected_order:
    got = startup_order.get(level, [])
    if not isinstance(got, list) or sorted(got) != sorted(expected_order[level]):
        order_ok = False
        break

if order_ok:
    score += 1
    print("[PASS] Startup order is correct")
    for level, components in startup_order.items():
        print(f"       {level}: {components}")
else:
    print("[FAIL] Expected:", expected_order)
    print("       Got:     ", startup_order)

## TODO 3: Identify the four primary data flows

In [ ]:
data_flows = {
    "request_path":     ["user", "fastapi"],
    "agent_to_llm":     ["langgraph", "llm_api"],
    "rag_retrieval":    ["langgraph", "chromadb"],
    "trace_collection": ["langgraph", "langfuse"],
}

In [ ]:
total += 1
expected_flows = {
    "request_path":     ["user", "fastapi"],
    "agent_to_llm":     ["langgraph", "llm_api"],
    "rag_retrieval":    ["langgraph", "chromadb"],
    "trace_collection": ["langgraph", "langfuse"],
}
if data_flows == expected_flows:
    score += 1
    print("[PASS] Data flows are correct")
    for name, flow in data_flows.items():
        print(f"       {name}: {flow[0]} -> {flow[1]}")
else:
    print("[FAIL] Expected:", expected_flows)
    print("       Got:     ", data_flows)

## TODO 4: Identify monitoring points for each component

In [ ]:
monitoring_points = {
    "fastapi":    ["request_latency", "error_rate", "request_count"],
    "langgraph":  ["step_duration", "tool_call_count", "agent_errors"],
    "chromadb":   ["query_latency", "collection_size", "retrieval_count"],
    "llm_api":    ["token_usage", "response_time", "cost_per_request"],
}

In [ ]:
total += 1
expected_monitoring = {
    "fastapi":    ["request_latency", "error_rate", "request_count"],
    "langgraph":  ["step_duration", "tool_call_count", "agent_errors"],
    "chromadb":   ["query_latency", "collection_size", "retrieval_count"],
    "llm_api":    ["token_usage", "response_time", "cost_per_request"],
}
if monitoring_points == expected_monitoring:
    score += 1
    print("[PASS] Monitoring points are correct")
    for comp, points in monitoring_points.items():
        print(f"       {comp}: {points}")
else:
    print("[FAIL] Expected:", expected_monitoring)
    print("       Got:     ", monitoring_points)

## TODO 5: Build the complete architecture document

In [ ]:
architecture_doc = {
    "project_name": "ai-support-agent",
    "components": component_roles,
    "startup_order": startup_order,
    "data_flows": data_flows,
    "monitoring_points": monitoring_points,
}

In [ ]:
total += 1
try:
    checks = [
        isinstance(architecture_doc, dict),
        architecture_doc.get("project_name") == "ai-support-agent",
        architecture_doc.get("components") == expected_roles,
        isinstance(architecture_doc.get("startup_order"), dict),
        isinstance(architecture_doc.get("data_flows"), dict),
        isinstance(architecture_doc.get("monitoring_points"), dict),
    ]
    if all(checks):
        score += 1
        out_path = os.path.join(WORKDIR, "architecture.json")
        with open(out_path, "w") as f:
            json.dump(architecture_doc, f, indent=2)
        print(f"[PASS] Architecture document saved to {out_path}")
    else:
        failed = [i for i, c in enumerate(checks) if not c]
        print(f"[FAIL] Architecture doc checks failed at indices: {failed}")
        print(f"       Got: {architecture_doc}")
except Exception as e:
    print(f"[FAIL] Architecture doc exception: {e}")

## Summary

In [ ]:
print(f"Lab 01 Score: {score}/{total}")